# Lab 1: Primes and Euclid leave trails
Use the sieve, Euclid's algorithm, and continued fractions to expose structure; then certify each pattern by hand.

## The sieve of Eratosthenes
Record not only which numbers are composite, but which prime first crosses each one out.

In [ ]:
from math import isqrt

def sieve_trace(N):
    alive = [True] * (N + 1)
    alive[0] = alive[1] = False
    crossed_by = {}
    for p in range(2, isqrt(N) + 1):
        if alive[p]:
            for m in range(p*p, N + 1, p):
                if alive[m]:
                    crossed_by[m] = p
                alive[m] = False
    primes = [n for n in range(2, N + 1) if alive[n]]
    return primes, crossed_by

primes, crossed_by = sieve_trace(100)
primes

**Prove the stopping rule.** Why must a composite $n\le N$ have a prime factor at most $\sqrt N$? Why is it safe to begin crossing out multiples of $p$ at $p^2$?

In [ ]:
def euclid_trace(a, b):
    rows = []
    while b:
        q, r = divmod(a, b)
        rows.append((a, q, b, r))
        a, b = b, r
    return rows, a

rows, d = euclid_trace(391, 299)
rows, d

**Task.** Explain why replacing `(a,b)` by `(b,r)` preserves every common divisor. Then use the trace to reconstruct Bezout coefficients.

In [ ]:
d, x, y = xgcd(391, 299)
d, x, y, 391*x + 299*y

## Continued fractions
Compare Euclid's quotients with the partial quotients below.

In [ ]:
q = QQ(415)/93
cf = continued_fraction(q)
cf, list(cf.convergents())

## Convergents are extended-Euclidean coefficients
For $A/B=[a_0;\ldots,a_n]$, compare the Euclidean remainder $r_k$ with $(-1)^k(q_kA-p_kB)$.

In [ ]:
A, B = 415, 93
rows, d = euclid_trace(A, B)
convs = list(continued_fraction(QQ(A)/B).convergents())
comparison = []
for k, c in enumerate(convs):
    p, q = c.numerator(), c.denominator()
    remainder_from_cf = (-1)^k * (q*A - p*B)
    comparison.append((k, rows[k][1], c, rows[k][3], remainder_from_cf))
comparison   # (index, partial quotient, convergent, Euclid remainder, CF formula)

**Task.** Prove the remainder formula from the two recurrences. Explain why the penultimate convergent supplies Bezout coefficients for the gcd.

In [ ]:
alpha = continued_fraction(sqrt(2))
conv = [alpha.convergent(k) for k in range(10)]
[(c, c.numerator()^2 - 2*c.denominator()^2) for c in conv]

In [ ]:
lower = conv[0::2]
upper = conv[1::2]
lower_increases = all(lower[i] < lower[i+1] for i in range(len(lower)-1))
upper_decreases = all(upper[i] > upper[i+1] for i in range(len(upper)-1))
[(k, c, 'below' if c < sqrt(2) else 'above') for k, c in enumerate(conv)], lower_increases, upper_decreases

**The Mobius derivative trick.** With $M_k(t)=(p_kt+p_{k-1})/(q_kt+q_{k-1})$, compute $M_k'(t)$. Use its alternating sign to prove that the zero-indexed even convergents increase below $x$, while the odd convergents decrease above $x$.

**Conjecture and prove.** Which indices solve $p^2-2q^2=1$, and which solve $-1$? Use the recurrence, not the decimal approximations.

**Extension.** Replace 2 by nonsquare values from 3 through 20. Record the repeating block and the first solution to $p^2-dq^2=1$.

## The matrix engine
Multiply the digit matrices $D(a)=\begin{pmatrix}a&1\\1&0\end{pmatrix}$. The two columns should be consecutive convergents, and the determinant should alternate between $1$ and $-1$.

In [ ]:
def convergent_matrix(coeffs):
    C = identity_matrix(ZZ, 2)
    for a in coeffs:
        C *= matrix(ZZ, [[a, 1], [1, 0]])
    return C

C = convergent_matrix([4, 2, 6, 7])
C, C.det(), C.inverse()

**Explain.** Why does the inverse still have integer entries? Locate the Bezout identity $13(415)-58(93)=1$ in the displayed matrices.

## Repeating base expansions are finite-state processes
Long division has only finitely many possible remainders. The first repeated remainder determines the repeating block.

In [ ]:
def base_expansion_trace(m, q, b=10, steps=30):
    remainder = m % q
    seen = {}
    digits = []
    trace = []
    for n in range(steps):
        if remainder in seen:
            return digits, trace, seen[remainder], n - seen[remainder]
        seen[remainder] = n
        digit, new_remainder = divmod(b*remainder, q)
        trace.append((n, remainder, digit, new_remainder))
        digits.append(digit)
        remainder = new_remainder
    return digits, trace, None, None

base_expansion_trace(1, 7), base_expansion_trace(5, 12)

**Compare two periodicities.** Decimal repetition comes from finitely many remainders. Continued-fraction repetition for a quadratic irrational comes from finitely many complete-quotient states. Verify Sage's expansions below, then use the matrix of one repeating block to recover a quadratic equation.

In [ ]:
continued_fraction(sqrt(2)), continued_fraction(1 + sqrt(2)), continued_fraction((1 + sqrt(5))/2)

## Periodicity meets Pell
A solution $u^2-Dv^2=1$ makes $U=\begin{pmatrix}u&Dv\\v&u\end{pmatrix}$ an $SL(2,\mathbb Z)$ matrix. Its powers multiply by powers of $u+v\sqrt D$ and generate further Pell solutions.

In [ ]:
D = 3
U = matrix(ZZ, [[2, 3], [1, 2]])
pell_rows = []
for n in range(1, 7):
    p, q = (U^n).column(0)
    pell_rows.append((n, p, q, p^2 - D*q^2, abs(sqrt(D) - QQ(p)/q), QQ(1)/(2*q^2)))
pell_rows   # the last two columns verify Legendre's strict inequality

**Prove both arrows.** Factor $|D-p^2/q^2|$ to show why every $\pm1$ Pell solution satisfies Legendre's bound. Then explain how the repeating period of $\sqrt D$ supplies the first generator matrix.

## A Khinchin--Levy experiment with $\pi$ and $e$
For $x=[a_0;a_1,a_2,\ldots]$, compute $G_n=(a_1\cdots a_n)^{1/n}$ and $B_n=q_n^{2/n}$. The almost-everywhere targets are about $2.685452$ and $10.731016$. Numerical agreement for a named constant is evidence, not proof.

In [ ]:
def khinchin_levy_rows(x, cutoffs):
    R = RealField(80)
    cf = continued_fraction(x)
    q_old, q_new = ZZ(1), ZZ(0)
    sum_log = R(0)
    rows = []
    for k in range(max(cutoffs) + 1):
        a = ZZ(cf[k])
        q_old, q_new = q_new, a*q_new + q_old
        if k > 0:
            sum_log += R(a).log()
        if k in cutoffs:
            G = (sum_log/k).exp()
            B = (2*R(q_new).log()/k).exp()
            rows.append((k, G, B, B.log()/R(10).log()))
    return rows

pi_rows = khinchin_levy_rows(pi, [100, 1000, 10000])
pi_rows   # n, G_n(pi), B_n(pi), decimal digits per CF term

**Why $e$ is different.** Euler's expansion is $e=[2;1,2,1,1,4,1,1,6,\ldots]$. For $n=3m$, the product of the noninitial partial quotients is exactly $2^m m!$, so $G_{3m}(e)\sim(2m/e)^{1/3}\to\infty$. The next cell displays this provable divergence.

In [ ]:
R = RealField(80)
[(3*m, ((m*R(2).log() + R(factorial(m)).log())/(3*m)).exp())
 for m in [10, 100, 1000, 10000]]

## Gosper's streaming arithmetic
A homographic state $M$ represents $(At+B)/(Ct+D)$. Absorbing an input digit multiplies $M$ on the right by its digit matrix; emitting a certified output digit multiplies on the left by $\begin{pmatrix}0&1\\1&-k\end{pmatrix}$. The following exact-rational calculation reproduces the first four output digits of $\pi/2$.

In [ ]:
def digit_matrix(m):
    return matrix(ZZ, [[m, 1], [1, 0]])

def absorb(M, m):
    return M * digit_matrix(m)

def emit(M, k):
    return matrix(ZZ, [[0, 1], [1, -k]]) * M

def mobius_endpoint_range(M):
    A, B, C, D = M.list()
    values = [QQ(A + B)/QQ(C + D), QQ(A)/QQ(C)]
    return min(values), max(values)

M = matrix(ZZ, [[1, 0], [0, 2]])       # x/2
M = absorb(absorb(M, 3), 7)            # pi begins [3;7,...]
rows = []
for _ in range(4):
    lo, hi = mobius_endpoint_range(M)
    k = floor(lo)
    assert floor(hi) == k               # sufficient in these four rows
    rows.append((k, M, lo, hi))
    M = emit(M, k)
rows, M                                 # digits 1,1,1,3; then state 2t

Now absorb the next two input digits, 15 and 1. The two endpoint values are 31 and 32, but the value 32 occurs only at the nonattained limit $t=\infty$. Thus the exact range is $[31,32)$ and the next output digit is 31. This is why a full implementation must retain open and closed endpoint information.

In [ ]:
M = absorb(absorb(M, 15), 1)
mobius_endpoint_range(M), M             # [31,32), so emit 31